<a href="https://colab.research.google.com/github/venezianof/booksum/blob/main/9Routermioipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9Router: Token Saver and Model Orchestrator
This notebook will explore the concepts described in the 9Router architecture.

In [1]:
# Define the tiers as described
tiers = {
    "Tier 1": "Subscription (Claude Code, Codex, GitHub Copilot)",
    "Tier 2": "Cheap (GLM, MiniMax)",
    "Tier 3": "Free (Kiro, Vertex)"
}

print("9Router Configuration Initialized.")
for tier, models in tiers.items():
    print(f"{tier}: {models}")

9Router Configuration Initialized.
Tier 1: Subscription (Claude Code, Codex, GitHub Copilot)
Tier 2: Cheap (GLM, MiniMax)
Tier 3: Free (Kiro, Vertex)


In [2]:
def route_request(quota_exhausted=False, budget_limited=False):
    """
    Simulates the 9Router logic:
    1. Tier 1 if quota is available.
    2. Tier 2 if Tier 1 is exhausted and budget is available.
    3. Tier 3 if Tier 2 is budget limited.
    """
    if not quota_exhausted:
        return f"Using {tiers['Tier 1']}"
    elif not budget_limited:
        return f"Using {tiers['Tier 2']}"
    else:
        return f"Using {tiers['Tier 3']}"

# Scenario: Tier 1 quota is exhausted, but we have budget for Tier 2
print(f"Scenario 1: {route_request(quota_exhausted=True, budget_limited=False)}")

# Scenario: All limits reached, falling back to free
print(f"Scenario 2: {route_request(quota_exhausted=True, budget_limited=True)}")

Scenario 1: Using Cheap (GLM, MiniMax)
Scenario 2: Using Free (Kiro, Vertex)


In [7]:
class BudgetMonitor:
    def __init__(self, limits):
        self.limits = limits  # Presupuesto total permitido por tier
        self.spending = {tier: 0.0 for tier in limits}  # Gasto actual

    def record_usage(self, tier, cost):
        if tier in self.spending:
            self.spending[tier] += cost
            print(f"[Monitor] Gasto en {tier} actualizado: ${self.spending[tier]:.4f} / ${self.limits[tier]:.2f}")

            # Alerta automática: Si queda menos del 10%
            remaining = self.limits[tier] - self.spending[tier]
            threshold = self.limits[tier] * 0.10
            if remaining < threshold and remaining > 0:
                print(f"[⚠️ ALERT] ¡Atención! El presupuesto para {tier} está por debajo del 10% (Restante: ${remaining:.4f})")
            elif remaining <= 0:
                print(f"[🚫 CRITICAL] Presupuesto para {tier} AGOTADO.")

    def is_budget_ok(self, tier):
        return self.spending.get(tier, 0) < self.limits.get(tier, 0)

# Configuración de límites (ejemplo)
limits = {"Tier 2": 5.0, "Tier 1": 50.0} # Tier 3 es gratis ($0)
monitor = BudgetMonitor(limits)

# Simulación de uso
monitor.record_usage("Tier 2", 0.0025)
print(f"¿Hay presupuesto para Tier 2?: {monitor.is_budget_ok('Tier 2')}")

[Monitor] Gasto en Tier 2 actualizado: $0.0025 / $5.00
¿Hay presupuesto para Tier 2?: True


In [4]:
def route_with_budget(quota_exhausted, monitor):
    if not quota_exhausted:
        return "Tier 1"

    if monitor.is_budget_ok("Tier 2"):
        return "Tier 2"

    return "Tier 3"

# Prueba de la lógica con el monitor
print(f"Ruta elegida: {route_with_budget(quota_exhausted=True, monitor=monitor)}")

Ruta elegida: Tier 2


## Simulación de RTK Token Saver
El RTK (Real-Time Kernel) Token Saver optimiza los mensajes enviados a la API (especialmente `tool_results`) para reducir costos sin perder contexto.

In [5]:
import random

def simulate_rtk_saver(raw_tokens):
    """
    Simula la reducción de tokens mediante técnicas de optimización RTK.
    Según tu diagrama, el ahorro es de aproximadamente 20-40%.
    """
    reduction_factor = random.uniform(0.20, 0.40)
    optimized_tokens = int(raw_tokens * (1 - reduction_factor))
    savings = raw_tokens - optimized_tokens
    return optimized_tokens, savings, reduction_factor * 100

# Ejemplo de uso con una solicitud de 10,000 tokens
raw_input = 10000
optimized, saved, percent = simulate_rtk_saver(raw_input)

print(f"--- Reporte RTK ---")
print(f"Tokens Originales: {raw_input}")
print(f"Tokens Optimizados: {optimized}")
print(f"Tokens Ahorrados: {saved} ({percent:.2f}%)")

--- Reporte RTK ---
Tokens Originales: 10000
Tokens Optimizados: 6013
Tokens Ahorrados: 3987 (39.87%)


In [6]:
import pandas as pd

# Costo hipotético por 1M de tokens en Tier 2
COST_PER_1M = 0.60

def simulate_batch_requests(num_requests=10):
    total_raw_tokens = 0
    total_optimized_tokens = 0
    total_cost_saved = 0.0

    results = []

    for i in range(num_requests):
        raw_tokens = random.randint(1000, 15000)
        optimized, saved, p = simulate_rtk_saver(raw_tokens)

        # Calcular costos
        cost_original = (raw_tokens / 1_000_000) * COST_PER_1M
        cost_optimized = (optimized / 1_000_000) * COST_PER_1M

        # Registrar en el monitor
        monitor.record_usage("Tier 2", cost_optimized)

        total_raw_tokens += raw_tokens
        total_optimized_tokens += optimized
        total_cost_saved += (cost_original - cost_optimized)

        results.append({
            "Req": i+1,
            "Original": raw_tokens,
            "Optimized": optimized,
            "Saved": saved,
            "% Savings": f"{p:.1f}%"
        })

    display(pd.DataFrame(results))
    print(f"\n--- Resumen Acumulado ---")
    print(f"Total Tokens Originales: {total_raw_tokens}")
    print(f"Total Tokens Optimizados: {total_optimized_tokens}")
    print(f"Ahorro Total en Tokens: {total_raw_tokens - total_optimized_tokens}")
    print(f"Dinero Ahorrado (Estimado Tier 2): ${total_cost_saved:.6f}")

simulate_batch_requests(10)

[Monitor] Gasto en Tier 2 actualizado: $0.0039 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0094 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0110 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0142 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0153 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0183 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0192 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0220 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0267 / $5.00
[Monitor] Gasto en Tier 2 actualizado: $0.0287 / $5.00


,Req,Original,Optimized,Saved,% Savings
0,1,3486,2295,1191,34.2%
1,2,12478,9256,3222,25.8%
2,3,3651,2597,1054,28.9%
3,4,8181,5418,2763,33.8%
4,5,2545,1789,756,29.7%
5,6,7294,5059,2235,30.6%
6,7,1904,1404,500,26.2%
7,8,6085,4694,1391,22.9%
8,9,10578,7800,2778,26.3%
9,10,4936,3437,1499,30.4%



--- Resumen Acumulado ---
Total Tokens Originales: 61138
Total Tokens Optimizados: 43749
Ahorro Total en Tokens: 17389
Dinero Ahorrado (Estimado Tier 2): $0.010433


### Test dell'Avviso di Soglia (10%)
Questa cella simula un utilizzo intensivo del Tier 2 per innescare l'avviso automatico implementato nella classe `BudgetMonitor`.

In [8]:
print(f"--- Simulazione Stress Test Budget ---")
# Forziamo un consumo elevato per arrivare vicini al limite di $5.00
# Ogni blocco simula un costo di $0.50
consumo_blocco = 0.50

for i in range(10):
    print(f"\nEsecuzione blocco di richieste #{i+1}")
    monitor.record_usage("Tier 2", consumo_blocco)

    if monitor.spending["Tier 2"] >= monitor.limits["Tier 2"]:
        break

--- Simulazione Stress Test Budget ---

Esecuzione blocco di richieste #1
[Monitor] Gasto en Tier 2 actualizado: $0.5025 / $5.00

Esecuzione blocco di richieste #2
[Monitor] Gasto en Tier 2 actualizado: $1.0025 / $5.00

Esecuzione blocco di richieste #3
[Monitor] Gasto en Tier 2 actualizado: $1.5025 / $5.00

Esecuzione blocco di richieste #4
[Monitor] Gasto en Tier 2 actualizado: $2.0025 / $5.00

Esecuzione blocco di richieste #5
[Monitor] Gasto en Tier 2 actualizado: $2.5025 / $5.00

Esecuzione blocco di richieste #6
[Monitor] Gasto en Tier 2 actualizado: $3.0025 / $5.00

Esecuzione blocco di richieste #7
[Monitor] Gasto en Tier 2 actualizado: $3.5025 / $5.00

Esecuzione blocco di richieste #8
[Monitor] Gasto en Tier 2 actualizado: $4.0025 / $5.00

Esecuzione blocco di richieste #9
[Monitor] Gasto en Tier 2 actualizado: $4.5025 / $5.00
[⚠️ ALERT] ¡Atención! El presupuesto para Tier 2 está por debajo del 10% (Restante: $0.4975)

Esecuzione blocco di richieste #10
[Monitor] Gasto en Tie

## Traduzione dei Formati (OpenAI ↔ Claude)
Per permettere l'orchestrazione fluida tra Tier diversi, 9Router deve tradurre i messaggi nel formato corretto per ogni provider.

In [9]:
class FormatTranslator:
    @staticmethod
    def openai_to_claude(messages):
        """Converte una lista di messaggi OpenAI in formato Anthropic/Claude."""
        claude_messages = []
        system_prompt = ""

        for msg in messages:
            role = msg["role"]
            content = msg["content"]

            if role == "system":
                system_prompt = content
            elif role == "user":
                claude_messages.append({"role": "user", "content": content})
            elif role == "assistant":
                claude_messages.append({"role": "assistant", "content": content})

        return system_prompt, claude_messages

    @staticmethod
    def claude_to_openai(claude_messages, system_prompt=""):
        """Converte messaggi Claude e system prompt in formato OpenAI."""
        openai_messages = []
        if system_prompt:
            openai_messages.append({"role": "system", "content": system_prompt})

        for msg in claude_messages:
            openai_messages.append({"role": msg["role"], "content": msg["content"]})

        return openai_messages

# Esempio di test
translator = FormatTranslator()
openai_sample = [
    {"role": "system", "content": "Sei un assistente utile."},
    {"role": "user", "content": "Ciao, chi sei?"}
]

system, claude = translator.openai_to_claude(openai_sample)
print("--- Conversione OpenAI -> Claude ---")
print(f"System Prompt: {system}")
print(f"Messaggi: {claude}")

--- Conversione OpenAI -> Claude ---
System Prompt: Sei un assistente utile.
Messaggi: [{'role': 'user', 'content': 'Ciao, chi sei?'}]


In [10]:
print("--- Conversione Inversa: Claude -> OpenAI ---")
# Utilizziamo le variabili 'claude' e 'system' generate nella cella precedente
openai_reconstructed = translator.claude_to_openai(claude, system_prompt=system)

print(f"Format OpenAI Ricostruito:\n{openai_reconstructed}")

# Verifica di integrità
if openai_reconstructed == openai_sample:
    print("\n✅ Test Superato: I formati corrispondono perfettamente.")
else:
    print("\n❌ Test Fallito: C'è una discrepanza tra l'originale e il ricostruito.")

--- Conversione Inversa: Claude -> OpenAI ---
Format OpenAI Ricostruito:
[{'role': 'system', 'content': 'Sei un assistente utile.'}, {'role': 'user', 'content': 'Ciao, chi sei?'}]

✅ Test Superato: I formati corrispondono perfettamente.


## Meccanismo di Auto Token Refresh
In un sistema reale, le quote e i budget vengono resettati periodicamente. Implementiamo una funzione di `refresh` per simulare questo comportamento.

In [11]:
def auto_token_refresh(monitor):
    """
    Simula il ripristino dei budget e delle quote.
    Pulisce lo spending registrato nel monitor.
    """
    print("--- Esecuzione Auto Token Refresh ---")
    for tier in monitor.spending:
        monitor.spending[tier] = 0.0
    print("✅ Tutte le quote e i budget sono stati resettati.")

# Dimostrazione del refresh dopo lo stress test precedente
print(f"Stato budget prima del refresh (Tier 2): ${monitor.spending['Tier 2']:.2f}")
auto_token_refresh(monitor)
print(f"Stato budget dopo il refresh (Tier 2): ${monitor.spending['Tier 2']:.2f}")
print(f"Possiamo usare di nuovo il Tier 2? {monitor.is_budget_ok('Tier 2')}")

Stato budget prima del refresh (Tier 2): $5.00
--- Esecuzione Auto Token Refresh ---
✅ Tutte le quote e i budget sono stati resettati.
Stato budget dopo il refresh (Tier 2): $0.00
Possiamo usare di nuovo il Tier 2? True


## Integrazione API Effettiva (Simulazione CLI)
Il sistema 9Router è progettato per interfacciarsi con uno strumento CLI in esecuzione locale. Di seguito i comandi per preparare l'ambiente e la logica di connessione.

### Requisiti Ambiente
Per avviare il backend del router, eseguire nel terminale:
```bash
npm run build
PORT=20128 HOSTNAME=0.0.0.0 NEXT_PUBLIC_BASE_URL=http://localhost:20128 npm run start
```

In [12]:
import requests
import json

class LocalCLIConnector:
    def __init__(self, endpoint="http://localhost:20128"):
        self.endpoint = endpoint

    def send_to_router(self, payload):
        """
        Simula l'invio del payload (già ottimizzato da RTK e tradotto dal FormatTranslator)
        all'endpoint locale della CLI.
        """
        print(f"[CLI] Tentativo di invio a {self.endpoint}...")
        # In un ambiente reale qui faremmo: requests.post(self.endpoint, json=payload)
        # Simuliamo il successo
        return {"status": "success", "response": "Risposta generata dal modello tramite 9Router CLI"}

# Esempio di flusso completo
cli = LocalCLIConnector()

# Payload ipotetico (già tradotto in formato Claude)
final_payload = {
    "model": "claude-3-opus",
    "messages": claude,
    "optimized_with_rtk": True
}

response = cli.send_to_router(final_payload)
print(f"Risposta Finale: {response['response']}")

[CLI] Tentativo di invio a http://localhost:20128...
Risposta Finale: Risposta generata dal modello tramite 9Router CLI


## Storico Chiamate (Logging)
Implementiamo un sistema di logging per tracciare tutte le interazioni con l'orchestratore e monitorare le prestazioni nel tempo.

In [15]:
from datetime import datetime
import pandas as pd

class CallLogger:
    def __init__(self):
        self.history = []

    def log_call(self, endpoint, payload, response):
        entry = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "endpoint": endpoint,
            "model": payload.get("model", "unknown"),
            "optimized": payload.get("optimized_with_rtk", False),
            "status": response.get("status", "error")
        }
        self.history.append(entry)
        print(f"[Logger] Chiamata registrata: {entry['timestamp']} - Modello: {entry['model']}")

    def display_history(self):
        if not self.history:
            print("Nessuna chiamata registrata.")
            return
        df_history = pd.DataFrame(self.history)
        display(df_history)

# Inizializziamo il logger globale con un nome valido
router_logger = CallLogger()

Aggiorniamo il `LocalCLIConnector` per integrare il logging automatico.

In [16]:
class LocalCLIConnectorWithLogging(LocalCLIConnector):
    def __init__(self, logger, endpoint="http://localhost:20128"):
        super().__init__(endpoint)
        self.logger = logger

    def send_to_router(self, payload):
        # Chiamata alla logica di base
        response = super().send_to_router(payload)
        # Registrazione nel log
        self.logger.log_call(self.endpoint, payload, response)
        return response

# Test del nuovo connettore con logging
cli_logged = LocalCLIConnectorWithLogging(router_logger)
cli_logged.send_to_router(final_payload)

print("\n--- Visualizzazione Storico ---")
router_logger.display_history()

[CLI] Tentativo di invio a http://localhost:20128...
[Logger] Chiamata registrata: 2026-05-11 04:56:57 - Modello: claude-3-opus

--- Visualizzazione Storico ---


,timestamp,endpoint,model,optimized,status
0,2026-05-11 04:56:57,http://localhost:20128,claude-3-opus,True,success


## Analisi Strategia: maximize-claude
In base alla configurazione fornita, ecco come 9Router gestirà le risorse:

1.  **Tier 1 (Main)**: `cc/claude-opus-4-7` - Utilizzo prioritario della sottoscrizione.
2.  **Tier 2 (Backup)**: `glm/glm-5.1` - Interviene automaticamente all'esaurimento della quota Claude.
3.  **Tier 3 (Emergency)**: `kr/claude-sonnet-4.5` - Fallback gratuito per garantire la continuità del servizio.

**Costi Previsti:** $20 (Sub) + ~$5 (Usage) = **$25/mese**.

In [17]:
# Aggiornamento finale dei modelli nel sistema per riflettere la strategia 'maximize-claude'
tiers["Tier 1"] = "cc/claude-opus-4-7 (Subscription)"
tiers["Tier 2"] = "glm/glm-5.1 (Cheap Backup)"
tiers["Tier 3"] = "kr/claude-sonnet-4.5 (Free Fallback)"

print("Configurazione 'maximize-claude' applicata con successo.")
display(pd.DataFrame.from_dict(tiers, orient='index', columns=['Modello Selezionato']))

Configurazione 'maximize-claude' applicata con successo.


,Modello Selezionato
Tier 1,cc/claude-opus-4-7 (Subscription)
Tier 2,glm/glm-5.1 (Cheap Backup)
Tier 3,kr/claude-sonnet-4.5 (Free Fallback)


## Analisi Strategia: free-forever
Questa configurazione mira a mantenere i costi a **$0** sfruttando le API gratuite e i modelli senza autenticazione, mantenendo comunque un'alta qualità di risposta.

1. **Tier 1 (Main)**: `kr/claude-sonnet-4.5` - Qualità Claude 4.5 gratuita e illimitata.
2. **Tier 2 (Backup)**: `kr/glm-5` - Backup gratuito tramite Kiro.
3. **Tier 3 (Emergency)**: `oc/<auto>` - OpenCode Free, utilizzabile senza autenticazione.

**Costi Previsti:** **$0/mese**.

In [18]:
# Configurazione per la strategia 'free-forever'
free_tiers = {
    "Tier 1": "kr/claude-sonnet-4.5 (Free Unlimited)",
    "Tier 2": "kr/glm-5 (Kiro Free)",
    "Tier 3": "oc/<auto> (OpenCode No-Auth)"
}

# Simuliamo l'applicazione di questa combo
print("Configurazione 'free-forever' applicata.")
display(pd.DataFrame.from_dict(free_tiers, orient='index', columns=['Modello Selezionato']))

# Verifichiamo il logger con un payload gratuito
free_payload = {
    "model": "kr/claude-sonnet-4.5",
    "messages": claude,
    "optimized_with_rtk": True
}

cli_logged.send_to_router(free_payload)
print("\n--- Storico Aggiornato ---")
router_logger.display_history()

Configurazione 'free-forever' applicata.


,Modello Selezionato
Tier 1,kr/claude-sonnet-4.5 (Free Unlimited)
Tier 2,kr/glm-5 (Kiro Free)
Tier 3,oc/<auto> (OpenCode No-Auth)


[CLI] Tentativo di invio a http://localhost:20128...
[Logger] Chiamata registrata: 2026-05-11 04:59:16 - Modello: kr/claude-sonnet-4.5

--- Storico Aggiornato ---


,timestamp,endpoint,model,optimized,status
0,2026-05-11 04:56:57,http://localhost:20128,claude-3-opus,True,success
1,2026-05-11 04:59:16,http://localhost:20128,kr/claude-sonnet-4.5,True,success


## Analisi Strategia: always-on
Questa strategia implementa 5 livelli di fallback per garantire **zero downtime**.

1. **Layer 1**: `cc/claude-opus-4-7` (Best Quality - Subscription)
2. **Layer 2**: `cx/gpt-5.5` (Second Subscription)
3. **Layer 3**: `glm/glm-5.1` (Cheap, resets daily)
4. **Layer 4**: `minimax/MiniMax-M2.7` (Cheapest, 5h reset)
5. **Layer 5**: `kr/claude-sonnet-4.5` (Free Unlimited)

In [19]:
# Configurazione per la strategia 'always-on'
always_on_layers = {
    "Layer 1 (Primary)": "cc/claude-opus-4-7",
    "Layer 2 (Secondary Sub)": "cx/gpt-5.5",
    "Layer 3 (Cheap Backup)": "glm/glm-5.1",
    "Layer 4 (Micro Backup)": "minimax/MiniMax-M2.7",
    "Layer 5 (Emergency Free)": "kr/claude-sonnet-4.5"
}

print("Configurazione 'always-on' inizializzata.")
display(pd.DataFrame.from_dict(always_on_layers, orient='index', columns=['Modello Selezionato']))

# Simulazione di una chiamata tramite il logger per confermare la tracciabilità
always_on_payload = {
    "model": "cc/claude-opus-4-7",
    "messages": claude,
    "optimized_with_rtk": True
}

cli_logged.send_to_router(always_on_payload)
print("\n--- Storico con Strategia Always-On ---")
router_logger.display_history()

Configurazione 'always-on' inizializzata.


,Modello Selezionato
Layer 1 (Primary),cc/claude-opus-4-7
Layer 2 (Secondary Sub),cx/gpt-5.5
Layer 3 (Cheap Backup),glm/glm-5.1
Layer 4 (Micro Backup),minimax/MiniMax-M2.7
Layer 5 (Emergency Free),kr/claude-sonnet-4.5


[CLI] Tentativo di invio a http://localhost:20128...
[Logger] Chiamata registrata: 2026-05-11 05:00:51 - Modello: cc/claude-opus-4-7

--- Storico con Strategia Always-On ---


,timestamp,endpoint,model,optimized,status
0,2026-05-11 04:56:57,http://localhost:20128,claude-3-opus,True,success
1,2026-05-11 04:59:16,http://localhost:20128,kr/claude-sonnet-4.5,True,success
2,2026-05-11 05:00:51,http://localhost:20128,cc/claude-opus-4-7,True,success


## Stima dei Costi Mensili: Strategia 'always-on'

La strategia si basa su un mix di costi fissi (Abbonamenti) e costi variabili (Pay-as-you-go per i backup).

### Componenti di Costo:
1. **Layer 1 & 2 (Subscriptions):** Claude Code e GPT-5.5 (Stima: $20/mese ciascuno).
2. **Layer 3 & 4 (Cheap/Micro):** Budget allocato per coprire i picchi quando le sottoscrizioni sono sature.
3. **Layer 5 (Free):** $0.

In [20]:
# Parametri di costo
subs_cost = 20.0 + 20.0  # Claude + GPT
backup_budget = 15.0      # Budget variabile per GLM/MiniMax

total_monthly_estimate = subs_cost + backup_budget

cost_data = {
    "Voce di Spesa": ["Abbonamenti Premium (L1, L2)", "Budget Backup (L3, L4)", "Fallback Free (L5)", "TOTALE STIMATO"],
    "Costo Mensile ($)": [subs_cost, backup_budget, 0.0, total_monthly_estimate],
    "Tipo": ["Fisso", "Variabile", "Gratis", "Investimento Totale"]
}

df_costs = pd.DataFrame(cost_data)

print(f"--- Analisi Economica Strategia Always-On ---")
display(df_costs)

print(f"\nInvestimento minimo garantito: ${subs_cost:.2f}/mese")
print(f"Flessibilità massima con backup: ${total_monthly_estimate:.2f}/mese")

--- Analisi Economica Strategia Always-On ---


,Voce di Spesa,Costo Mensile ($),Tipo
0,"Abbonamenti Premium (L1, L2)",40.0,Fisso
1,"Budget Backup (L3, L4)",15.0,Variabile
2,Fallback Free (L5),0.0,Gratis
3,TOTALE STIMATO,55.0,Investimento Totale



Investimento minimo garantito: $40.00/mese
Flessibilità massima con backup: $55.00/mese


## Analisi Strategia: openclaw-free
Questa combo è ottimizzata per l'integrazione con piattaforme di messaggistica e garantisce un costo di gestione nullo.

1. **Tier 1**: `kr/claude-sonnet-4.5` (Claude 4.5 Free)
2. **Tier 2**: `kr/glm-5` (GLM-5 Free)
3. **Tier 3**: `kr/MiniMax-M2.5` (MiniMax Free)

**Costo Mensile:** $0
**Accesso:** WhatsApp, Telegram, Slack, Discord, iMessage, Signal.

In [21]:
# Configurazione 'openclaw-free'
openclaw_tiers = {
    "Primary": "kr/claude-sonnet-4.5",
    "Secondary": "kr/glm-5",
    "Tertiary": "kr/MiniMax-M2.5"
}

print("Configurazione 'openclaw-free' applicata.")
display(pd.DataFrame.from_dict(openclaw_tiers, orient='index', columns=['Modello (Versione Free)']))

# Simulazione di instradamento per questa specifica combo
openclaw_payload = {
    "model": "kr/claude-sonnet-4.5",
    "messages": claude,
    "platform_origin": "WhatsApp",
    "optimized_with_rtk": True
}

cli_logged.send_to_router(openclaw_payload)
print("\n--- Registro Chiamate con Origine Piattaforma ---")
router_logger.display_history()

Configurazione 'openclaw-free' applicata.


,Modello (Versione Free)
Primary,kr/claude-sonnet-4.5
Secondary,kr/glm-5
Tertiary,kr/MiniMax-M2.5


[CLI] Tentativo di invio a http://localhost:20128...
[Logger] Chiamata registrata: 2026-05-11 05:02:11 - Modello: kr/claude-sonnet-4.5

--- Registro Chiamate con Origine Piattaforma ---


,timestamp,endpoint,model,optimized,status
0,2026-05-11 04:56:57,http://localhost:20128,claude-3-opus,True,success
1,2026-05-11 04:59:16,http://localhost:20128,kr/claude-sonnet-4.5,True,success
2,2026-05-11 05:00:51,http://localhost:20128,cc/claude-opus-4-7,True,success
3,2026-05-11 05:02:11,http://localhost:20128,kr/claude-sonnet-4.5,True,success


## Confronto Prestazioni e Costi: Always-On vs. OpenClaw-Free

Analizziamo le due strategie basandoci sui dati simulati finora:

| Caratteristica | Always-On | OpenClaw-Free |
| :--- | :--- | :--- |
| **Costo Mensile Stimato** | ~$55.00 | **$0.00** |
| **Livelli di Ridondanza** | 5 Layer (Fallback profondo) | 3 Tier (Solo Free) |
| **Affidabilità** | Massima (SLA garantita dai sub) | Best-effort (Basata su quote free) |
| **Target Utente** | Produzione / Business Critical | Uso Personale / Automazione Social |
| **Modello di Punta** | Claude Opus 4.7 / GPT 5.5 | Claude Sonnet 4.5 (Free) |

In [22]:
import pandas as pd

comparison_data = {
    "Strategia": ["Always-On", "OpenClaw-Free"],
    "Costo Fisso ($)": [40.0, 0.0],
    "Costo Variabile ($)": [15.0, 0.0],
    "Totale ($)": [55.0, 0.0],
    "Fallback Tiers": [5, 3],
    "Priorità": ["Qualità/Disponibilità", "Risparmio/Integrazione"]
}

df_comp = pd.DataFrame(comparison_data)

print("--- Confronto Analitico 9Router ---")
display(df_comp)

# Calcolo del risparmio annuo passando a OpenClaw-Free
risparmio_annuo = df_comp.loc[0, 'Totale ($)'] * 12
print(f"\n💡 Il passaggio a 'OpenClaw-Free' genera un risparmio potenziale di ${risparmio_annuo:.2f} all'anno.")

--- Confronto Analitico 9Router ---


,Strategia,Costo Fisso ($),Costo Variabile ($),Totale ($),Fallback Tiers,Priorità
0,Always-On,40.0,15.0,55.0,5,Qualità/Disponibilità
1,OpenClaw-Free,0.0,0.0,0.0,3,Risparmio/Integrazione



💡 Il passaggio a 'OpenClaw-Free' genera un risparmio potenziale di $660.00 all'anno.


## Esempio di Chiamata API Reale (CLI Backend)

Per inviare una richiesta effettiva al backend locale (una volta avviato), utilizziamo la libreria `requests`. Ecco il formato standard atteso dall'endpoint `/v1/chat/completions`.

In [23]:
import requests
import json

def call_9router_api(model="cc/claude-opus-4-6", prompt="Spiega brevemente il concetto di tiered routing."):
    url = "http://localhost:20128/v1/chat/completions"
    headers = {
        "Authorization": "Bearer your-api-key",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": False # Impostato a False per semplicità in questo esempio
    }

    print(f"Invio richiesta a {url}...")
    try:
        # Nota: Questo fallirà se il server locale non è in esecuzione
        # response = requests.post(url, headers=headers, json=payload, timeout=5)
        # response.raise_for_status()
        # return response.json()
        print("\n[Simulazione] Payload JSON inviato correttamente:")
        print(json.dumps(payload, indent=2))
    except Exception as e:
        print(f"Errore di connessione (Backend non rilevato): {e}")

call_9router_api()

Invio richiesta a http://localhost:20128/v1/chat/completions...

[Simulazione] Payload JSON inviato correttamente:
{
  "model": "cc/claude-opus-4-6",
  "messages": [
    {
      "role": "user",
      "content": "Spiega brevemente il concetto di tiered routing."
    }
  ],
  "stream": false
}


### Resource Discovery: Models & Combos
9Router provides a standard `/v1/models` endpoint. This allows clients to dynamically discover available model tiers and pre-configured strategies (combos).

In [25]:
def get_9router_resources():
    url = "http://localhost:20128/v1/models"
    headers = {"Authorization": "Bearer your-api-key"}

    print(f"GET {url}")

    # Simulated response from the local proxy
    simulated_resources = {
        "object": "list",
        "data": [
            {"id": "cc/claude-opus-4-7", "object": "model", "owned_by": "9router-tier1"},
            {"id": "kr/claude-sonnet-4.5", "object": "model", "owned_by": "9router-free"},
            {"id": "strategy/always-on", "object": "combo", "owned_by": "9router-orchestrator"},
            {"id": "strategy/free-forever", "object": "combo", "owned_by": "9router-orchestrator"}
        ]
    }

    print("\n[Simulazione] Risorse disponibili nel Backend:")
    display(pd.DataFrame(simulated_resources['data']))
    return simulated_resources

_ = get_9router_resources()

GET http://localhost:20128/v1/models

[Simulazione] Risorse disponibili nel Backend:


,id,object,owned_by
0,cc/claude-opus-4-7,model,9router-tier1
1,kr/claude-sonnet-4.5,model,9router-free
2,strategy/always-on,combo,9router-orchestrator
3,strategy/free-forever,combo,9router-orchestrator


## Simulazione API Streaming
Per testare la risposta in streaming, simuliamo l'arrivo di 'chunks' (frammenti di testo) dal server locale.

In [24]:
import time
import sys

def simulate_streaming_9router(model="cc/claude-opus-4-6", prompt="Scrivi una breve poesia sul codice."):
    print(f"--- Inizio Stream da: {model} ---\n")

    # Testo simulato che verrebbe dal backend
    simulated_response = """Il codice fluisce come un fiume d'argento,
Logica pura in ogni momento.
Righe di testo, bit nel vento,
Costruiamo il mondo con sentimento."""

    # Simulazione della ricezione a pezzi (chunks)
    for word in simulated_response.split(' '):
        sys.stdout.write(word + ' ')
        sys.stdout.flush()
        time.sleep(0.2) # Ritardo artificiale per simulare la latenza di rete

    print(f"\n\n--- Fine Stream ---")

    # Registrazione dell'evento nel logger (anche se è uno stream)
    log_entry = {"model": model, "optimized_with_rtk": True}
    cli_logged.logger.log_call("http://localhost:20128/v1/chat/completions", log_entry, {"status": "success"})

simulate_streaming_9router()

--- Inizio Stream da: cc/claude-opus-4-6 ---

Il codice fluisce come un fiume d'argento,
Logica pura in ogni momento.
Righe di testo, bit nel vento,
Costruiamo il mondo con sentimento. 

--- Fine Stream ---
[Logger] Chiamata registrata: 2026-05-11 05:05:11 - Modello: cc/claude-opus-4-6


## Real-Time Budget Monitoring Script
This script monitors spending by simulating a continuous flow of API traffic. It accounts for RTK optimization savings and checks against the defined budget limits in real-time.

In [26]:
import time

def run_realtime_monitor(iterations=5, tier="Tier 2"):
    print(f"--- Starting Real-Time Budget Monitor for {tier} ---")
    print(f"Budget Limit: ${monitor.limits.get(tier, 0):.2f}\n")

    for i in range(iterations):
        # Simulate a random request size
        raw_tokens = random.randint(5000, 20000)

        # Apply RTK optimization
        optimized, saved, p = simulate_rtk_saver(raw_tokens)

        # Calculate cost (using Tier 2 rates as example)
        cost = (optimized / 1_000_000) * COST_PER_1M

        print(f"[Request {i+1}] Original: {raw_tokens} | Optimized: {optimized} ({p:.1f}% saved)")

        # Record usage in the monitor
        monitor.record_usage(tier, cost)

        # Check if we should stop due to budget exhaustion
        if not monitor.is_budget_ok(tier):
            print(f"\n[🛑 STOP] Budget for {tier} exhausted. Halting requests.")
            break

        time.sleep(1) # Wait 1 second between 'real-time' calls

    print(f"\n--- Monitoring Session Finished ---")
    print(f"Final Spending for {tier}: ${monitor.spending[tier]:.4f}")

# Reset monitor before test to see the progression clearly
auto_token_refresh(monitor)
run_realtime_monitor(iterations=8)

--- Esecuzione Auto Token Refresh ---
✅ Tutte le quote e i budget sono stati resettati.
--- Starting Real-Time Budget Monitor for Tier 2 ---
Budget Limit: $5.00

[Request 1] Original: 15569 | Optimized: 9992 (35.8% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0060 / $5.00
[Request 2] Original: 8747 | Optimized: 6861 (21.6% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0101 / $5.00
[Request 3] Original: 18683 | Optimized: 11853 (36.6% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0172 / $5.00
[Request 4] Original: 14369 | Optimized: 9386 (34.7% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0229 / $5.00
[Request 5] Original: 18148 | Optimized: 14277 (21.3% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0314 / $5.00
[Request 6] Original: 13459 | Optimized: 8867 (34.1% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0367 / $5.00
[Request 7] Original: 17095 | Optimized: 10482 (38.7% saved)
[Monitor] Gasto en Tier 2 actualizado: $0.0430 / $5.00
[Request 8] Original: 13191 | O

## 📊 Report Finale della Sessione 9Router
Questo blocco genera un riepilogo consolidato di tutte le metriche di performance e di costo simulate.

In [27]:
def generate_final_report(monitor, logger):
    print("=== 9ROUTER SESSION SUMMARY ===\n")

    # 1. Budget Status
    print("--- Stato Budget Finale ---")
    budget_summary = []
    for t in monitor.limits:
        spent = monitor.spending.get(t, 0)
        limit = monitor.limits[t]
        budget_summary.append({
            "Tier": t,
            "Speso ($)": f"${spent:.4f}",
            "Limite ($)": f"${limit:.2f}",
            "Rimanente (%)": f"{(1 - spent/limit)*100:.1f}%" if limit > 0 else "N/A"
        })
    display(pd.DataFrame(budget_summary))

    # 2. Call History Analysis
    print("\n--- Analisi Traffico ---")
    if logger.history:
        df_log = pd.DataFrame(logger.history)
        counts = df_log['model'].value_counts().reset_index()
        counts.columns = ['Modello', 'Numero di Chiamate']
        display(counts)

        optimized_count = df_log['optimized'].sum()
        total_calls = len(df_log)
        print(f"Efficienza RTK: {optimized_count}/{total_calls} chiamate ottimizzate ({(optimized_count/total_calls)*100:.1f}%)")
    else:
        print("Nessun dato nel log.")

    # 3. Final Conclusion
    print("\n--- Conclusione ---")
    print("Tutti i dati della simulazione sono stati salvati nello stato del kernel.")

generate_final_report(monitor, router_logger)

=== 9ROUTER SESSION SUMMARY ===

--- Stato Budget Finale ---


,Tier,Speso ($),Limite ($),Rimanente (%)
0,Tier 2,$0.0479,$5.00,99.0%
1,Tier 1,$0.0000,$50.00,100.0%



--- Analisi Traffico ---


,Modello,Numero di Chiamate
0,kr/claude-sonnet-4.5,2
1,claude-3-opus,1
2,cc/claude-opus-4-7,1
3,cc/claude-opus-4-6,1


Efficienza RTK: 5/5 chiamate ottimizzate (100.0%)

--- Conclusione ---
Tutti i dati della simulazione sono stati salvati nello stato del kernel.


## 💾 Esportazione Dati
Salvataggio dei risultati della simulazione su file per la persistenza e l'analisi esterna.

In [29]:
if 'router_logger' in globals() and router_logger.history:
    df_history_export = pd.DataFrame(router_logger.history)
    df_history_export.to_csv('9router_call_logs.csv', index=False)
    print("✅ Log delle chiamate salvati in: 9router_call_logs.csv")

if 'monitor' in globals():
    budget_data = []
    for t in monitor.limits:
        budget_data.append({"Tier": t, "Spent": monitor.spending.get(t, 0), "Limit": monitor.limits[t]})
    df_budget_export = pd.DataFrame(budget_data)
    df_budget_export.to_csv('9router_budget_status.csv', index=False)
    print("✅ Stato del budget salvato in: 9router_budget_status.csv")

✅ Log delle chiamate salvati in: 9router_call_logs.csv
✅ Stato del budget salvato in: 9router_budget_status.csv


## 💾 Esportazione Dati
Salvataggio dei risultati della simulazione su file per la persistenza.

In [28]:
# Esportazione dei log e del budget in CSV
if router_logger.history:
    df_history_export = pd.DataFrame(router_logger.history)
    df_history_export.to_csv('9router_call_logs.csv', index=False)
    print("✅ Log delle chiamate salvati in: 9router_call_logs.csv")

budget_data = []
for t in monitor.limits:
    budget_data.append({"Tier": t, "Spent": monitor.spending[t], "Limit": monitor.limits[t]})

df_budget_export = pd.DataFrame(budget_data)
df_budget_export.to_csv('9router_budget_status.csv', index=False)
print("✅ Stato del budget salvato in: 9router_budget_status.csv")

✅ Log delle chiamate salvati in: 9router_call_logs.csv
✅ Stato del budget salvato in: 9router_budget_status.csv
